# 06 — Gold Layer: fact_attendance (Factless Facts)

**Purpose:** Build two factless fact tables for student attendance tracking.

**Kimball fact type:** Factless (two tables)

| Table | Type | What it records |
|---|---|---|
| `fact_attendance_coverage` | Factless — Coverage | Every scheduled session slot (attended or not) |
| `fact_attendance_actual` | Factless — Activity | Only sessions the student actually attended |

**Why two tables?**
A single table with an `attended` flag cannot answer:
*"Which students are enrolled but have NEVER attended a single session?"*

To detect complete absence you need the set difference:
students in coverage who have NO matching rows in actual.
This is the Kimball coverage fact pattern.

**At-risk scenario in this dataset:**
Linh Nguyen (linh.nguyen@email.com) is enrolled and has 5 scheduled sessions
in coverage — but attended = 0 for every one of them.
She will appear in coverage but not in actual → flagged as at-risk.

## Parameters

In [ ]:
pipeline_run_date = "2024-01-15"

## Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType
from delta.tables import DeltaTable

run_date = pipeline_run_date
FACT_COVERAGE = "gold.fact_attendance_coverage"
FACT_ACTUAL   = "gold.fact_attendance_actual"


def table_exists(table_name):
    try:
        spark.table(table_name)
        return True
    except Exception:
        return False


def date_to_key(date_col):
    return F.when(
        date_col.isNotNull(),
        F.date_format(date_col, "yyyyMMdd").cast(IntegerType())
    ).otherwise(F.lit(0))


print(f"Pipeline run date: {run_date}")
print("Setup complete.")

## Step 1 — Build Attendance Base

Load `silver.attendance` and resolve all dimension surrogate keys.
The base DataFrame is used for both coverage and actual tables.

In [ ]:
silver_attendance = spark.table("silver.attendance")

# Dimension lookups
dim_student = spark.table("gold.dim_student").filter("_is_current = true").select(
    "student_sk", "student_id"
)
dim_course  = spark.table("gold.dim_course").select("course_sk", "course_id")
dim_campus  = spark.table("gold.dim_campus").select("campus_sk", "campus_id")
dim_intake  = spark.table("gold.dim_intake").select("intake_sk", "intake_id")

attendance_base = (
    silver_attendance
    .join(dim_student, silver_attendance.student_id == dim_student.student_id, how="left")
    .join(dim_course,  silver_attendance.course_code == dim_course.course_id,  how="left")
    .join(dim_campus,  silver_attendance.campus_code == dim_campus.campus_id,  how="left")
    .join(dim_intake,  silver_attendance.intake_code == dim_intake.intake_id,  how="left")
    .withColumn("session_date_key", date_to_key(F.col("session_date")))
    .select(
        silver_attendance.session_id,
        F.coalesce(dim_student.student_sk, F.lit(-1)).alias("student_sk"),
        F.coalesce(dim_course.course_sk,   F.lit(-1)).alias("course_sk"),
        F.coalesce(dim_campus.campus_sk,   F.lit(-1)).alias("campus_sk"),
        F.coalesce(dim_intake.intake_sk,   F.lit(-1)).alias("intake_sk"),
        F.col("session_date_key"),
        silver_attendance.session_type,
        silver_attendance.attended,
        F.current_timestamp().alias("_loaded_at"),
    )
)

print(f"Attendance base rows: {attendance_base.count()}")
attendance_base.show(truncate=False)

## Step 2 — Build fact_attendance_coverage

All scheduled session slots — regardless of whether the student attended.
This is the complete picture of what SHOULD have happened.

Every row in the source CSV goes here (attended = 0 or 1).
`record_count = 1` — presence of the row is the fact.

In [ ]:
incoming_coverage = (
    attendance_base
    .withColumn("record_count", F.lit(1))
    .drop("attended")
)

if not table_exists(FACT_COVERAGE):
    print("[fact_coverage] First run — creating table")
    w = Window.orderBy("session_id")
    first_load = incoming_coverage.withColumn(
        "attendance_sk", F.row_number().over(w).cast(IntegerType())
    )
    first_load.write.format("delta").mode("overwrite").saveAsTable(FACT_COVERAGE)
    print(f"[fact_coverage] Inserted {first_load.count()} rows")

else:
    print("[fact_coverage] Merging into existing table")
    existing = spark.table(FACT_COVERAGE)
    max_sk = existing.agg(F.max("attendance_sk")).collect()[0][0] or 0

    new_sessions = incoming_coverage.join(
        existing.select("session_id"), on="session_id", how="left_anti"
    )
    new_count = new_sessions.count()
    if new_count > 0:
        w = Window.orderBy("session_id")
        new_sessions = new_sessions.withColumn(
            "attendance_sk",
            (F.row_number().over(w) + F.lit(max_sk)).cast(IntegerType())
        )
        new_sessions.write.format("delta").mode("append").saveAsTable(FACT_COVERAGE)
    print(f"[fact_coverage] New rows inserted: {new_count}")

print(f"[fact_coverage] Total rows: {spark.table(FACT_COVERAGE).count()}")

## Step 3 — Build fact_attendance_actual

Only sessions the student actually attended (`attended = 1`).
This is the record of what DID happen.

Linh Nguyen has `attended = 0` for all her sessions — she will have
rows in coverage but ZERO rows in actual. This is the at-risk signal.

In [ ]:
incoming_actual = (
    attendance_base
    .filter(F.col("attended") == 1)  # only sessions actually attended
    .withColumn("record_count", F.lit(1))
    .drop("attended")
)

print(f"Actual attendance rows (attended=1 only): {incoming_actual.count()}")

if not table_exists(FACT_ACTUAL):
    print("[fact_actual] First run — creating table")
    w = Window.orderBy("session_id")
    first_load = incoming_actual.withColumn(
        "attendance_sk", F.row_number().over(w).cast(IntegerType())
    )
    first_load.write.format("delta").mode("overwrite").saveAsTable(FACT_ACTUAL)
    print(f"[fact_actual] Inserted {first_load.count()} rows")

else:
    print("[fact_actual] Merging into existing table")
    existing = spark.table(FACT_ACTUAL)
    max_sk = existing.agg(F.max("attendance_sk")).collect()[0][0] or 0

    new_sessions = incoming_actual.join(
        existing.select("session_id"), on="session_id", how="left_anti"
    )
    new_count = new_sessions.count()
    if new_count > 0:
        w = Window.orderBy("session_id")
        new_sessions = new_sessions.withColumn(
            "attendance_sk",
            (F.row_number().over(w) + F.lit(max_sk)).cast(IntegerType())
        )
        new_sessions.write.format("delta").mode("append").saveAsTable(FACT_ACTUAL)
    print(f"[fact_actual] New rows inserted: {new_count}")

print(f"[fact_actual] Total rows: {spark.table(FACT_ACTUAL).count()}")

## Verification

### Coverage vs Actual counts

Coverage should always be >= Actual.
The difference = scheduled sessions not attended.

In [ ]:
coverage_count = spark.table(FACT_COVERAGE).count()
actual_count   = spark.table(FACT_ACTUAL).count()

print(f"Coverage rows (all scheduled): {coverage_count}")
print(f"Actual rows   (attended only): {actual_count}")
print(f"Missed sessions:               {coverage_count - actual_count}")
print()
assert coverage_count >= actual_count, "FAIL: actual > coverage — data integrity issue"
print("PASS: coverage >= actual")

### At-Risk Student Detection

Students with rows in coverage but ZERO rows in actual.
Linh Nguyen should appear here — enrolled but never attended.

In [ ]:
coverage = spark.table(FACT_COVERAGE)
actual   = spark.table(FACT_ACTUAL)
dim_student_full = spark.table("gold.dim_student").filter("_is_current = true")
dim_course_full  = spark.table("gold.dim_course")

# Students in coverage with NO matching rows in actual
# left_anti join: keep only coverage rows that have no match in actual
at_risk_sks = (
    coverage
    .select("student_sk", "course_sk", "intake_sk")
    .distinct()
    .join(
        actual.select("student_sk", "course_sk", "intake_sk").distinct(),
        on=["student_sk", "course_sk", "intake_sk"],
        how="left_anti"
    )
)

at_risk = (
    at_risk_sks
    .join(dim_student_full.select("student_sk", "student_id", "student_first_name", "student_last_name"),
          on="student_sk", how="left")
    .join(dim_course_full.select("course_sk", "course_name"),
          on="course_sk", how="left")
)

at_risk_count = at_risk.count()
print(f"At-risk students (enrolled, never attended): {at_risk_count}")
print()
at_risk.select(
    "student_id", "student_first_name", "student_last_name", "course_name"
).show(truncate=False)

# Linh Nguyen must appear
linh_at_risk = at_risk.filter(
    F.col("student_id").contains("linh.nguyen")
).count()
print(f"Linh Nguyen flagged as at-risk: {'PASS' if linh_at_risk > 0 else 'FAIL'}")

### Attendance Rate by Student

In [ ]:
# Coverage count per student
coverage_by_student = (
    coverage
    .groupBy("student_sk")
    .agg(F.count("*").alias("scheduled_sessions"))
)

# Actual count per student
actual_by_student = (
    actual
    .groupBy("student_sk")
    .agg(F.count("*").alias("attended_sessions"))
)

attendance_rate = (
    coverage_by_student
    .join(actual_by_student, on="student_sk", how="left")
    .join(dim_student_full.select("student_sk", "student_first_name", "student_last_name"),
          on="student_sk", how="left")
    .fillna(0, subset=["attended_sessions"])
    .withColumn(
        "attendance_rate_pct",
        F.round(
            F.col("attended_sessions") * 100.0 / F.col("scheduled_sessions"), 1
        )
    )
    .select(
        "student_first_name", "student_last_name",
        "scheduled_sessions", "attended_sessions", "attendance_rate_pct"
    )
    .orderBy("attendance_rate_pct")
)

print("Attendance rate by student:")
attendance_rate.show(truncate=False)

## Summary

Both factless fact tables built and verified.

**What was demonstrated:**
- `fact_attendance_coverage` — 34 rows (all scheduled sessions)
- `fact_attendance_actual` — fewer rows (attended sessions only)
- At-risk detection via `left_anti` join — Linh Nguyen correctly flagged
- Attendance rate calculated per student

**The Kimball factless fact insight:**
Neither table has traditional numeric measures — the presence or absence
of a row IS the information. This is the coverage fact pattern:
you cannot detect absence from a single table.

**All Gold notebooks complete.**
Next step: Connect Power BI via Direct Lake and build the semantic model.